# Grid MiniYOLO 資料

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/07-data/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.2.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""One deterministic scene, variable annotation counts, no downloads."""
import torch
from miniyolo.data import ShapeDataset, collate


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    image = torch.zeros(3, 64, 64)
    image[0, 12:28, 8:24] = 1
    image[2, 36:52, 40:56] = 1
    target = {"boxes": torch.tensor([[8., 12., 24., 28.], [40., 36., 56., 52.]]),
              "labels": torch.tensor([0, 1], dtype=torch.long)}
    expected_pixels = torch.zeros_like(image)
    for box, label in zip(target["boxes"], target["labels"]):
        x1, y1, x2, y2 = box.to(torch.long).tolist()
        channel = {0: 0, 1: 2}[label.item()]
        expected_pixels[channel, y1:y2, x1:x2] = 1
    assert torch.equal(image, expected_pixels), "Annotation and colored pixels disagree"
    empty = {"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}
    images, targets = collate([(image, target), (torch.zeros_like(image), empty)])
    assert images.shape == (2, 3, 64, 64)
    assert targets[1]["boxes"].shape == (0, 4)
    assert images[0, 0, 12:28, 8:24].sum().item() == 256
    assert images[0, 2, 36:52, 40:56].sum().item() == 256
    generated = ShapeDataset(n=8, size=64, seed=7)
    for im, ann in generated:
        assert im.dtype == torch.float32 and im.shape == (3, 64, 64)
        assert ann["boxes"].shape == (len(ann["labels"]), 4)
        boxes = ann["boxes"]
        assert bool(((boxes >= 0) & (boxes <= 64)).all())
        assert bool((boxes[:, 2:] > boxes[:, :2]).all())
    print("batch", tuple(images.shape), "counts", [len(t["boxes"]) for t in targets])
    print("red/blue channel sums", images[0, 0].sum().item(), images[0, 2].sum().item())
    print("dataset contract checked: 8 images")


if __name__ == '__main__':
    main()


batch (2, 3, 64, 64) counts [2, 0]
red/blue channel sums 256.0 256.0
dataset contract checked: 8 images
